In [ ]:
import sys
from pathlib import Path
carpeta_pipeline = next(ruta for ruta in [Path.cwd(), Path.cwd() / 'pipeline_ingesta', Path.cwd() / 'notebook/unidad_IV/pipeline_ingesta'] if (ruta / 'auditoria_pipeline.py').is_file())
sys.path.insert(0, str(carpeta_pipeline.resolve()))
from auditoria_pipeline import iniciar_etapa

auditoria_etapa = iniciar_etapa('04_staging', 'invoices')

# 04 - Staging de Invoices

Ejecutar despues de `02_profiling_contratos_Invoices.ipynb` y `03_cuarentena_reproceso_Invoices.ipynb`.

**Entrada:** `raw.invoices`, excluyendo `quarantine.invoices` y errores criticos de `raw.validacion_calidad_invoices` por `(id_raw, lote_id)`.

**Salida:** `staging.invoices`, con identificadores de factura, cliente y vendedor, fecha, nota de credito y totales de articulos. Se conservan `id_raw` y `lote_id` para trazabilidad.

Cada carga vuelve a comprobar las exclusiones y retira registros previamente cargados que ahora esten rechazados. Una cuarentena historica sigue bloqueando el registro, incluso si cambia su estado. Al finalizar se verifica que no haya rechazados en staging.

In [1]:
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, SERVIDOR, BD_ORIGEN, BD_DESTINO

engine_origen, engine_destino = obtener_engines()
print(f'Conexiones establecidas:')
print(f'  Servidor: {SERVIDOR}')
print(f'  Origen: {BD_ORIGEN}')
print(f'  Destino: {BD_DESTINO}')

Conexion OK (origen): WideWorldImporters
Conexion OK (destino): WideWorldImportersDW2
Conexiones establecidas:
  Servidor: localhost\SQLEXPRESS02
  Origen: WideWorldImporters
  Destino: WideWorldImportersDW2


## 1. Leer cabeceras RAW habilitadas de raw.invoices

In [ ]:
import pandas as pd
import json
from sqlalchemy import text

query_raw = """
SELECT r.id_raw, r.lote_id, r.payload
FROM raw.invoices r
WHERE NOT EXISTS (
    SELECT 1 FROM quarantine.invoices q
    WHERE q.id_raw = r.id_raw AND q.lote_id = r.lote_id
)
AND NOT EXISTS (
    SELECT 1 FROM raw.validacion_calidad_invoices v
    WHERE v.id_raw = r.id_raw AND v.lote_id = r.lote_id
      AND v.es_critico = 1
)
ORDER BY r.id_raw
"""

df_raw = pd.read_sql_query(query_raw, engine_destino)
print(f'Cabeceras habilitadas para staging: {len(df_raw)}')

Total de registros en raw.invoice_lines: 100
Primer payload (primeros 200 caracteres):
  {"ExtendedPrice": "2645.0", "InvoiceID": "1", "InvoiceLineID": "1", "LastEditedWhen": "2013-01-01T12:00:00", "Quantity": "10", "StockItemID": "67", "TaxAmount": "345.0", "TaxRate": "15.0", "UnitPrice"...


## 2. Descomprimir payloads JSON

In [ ]:
datos_lista = []
for idx, row in df_raw.iterrows():
    payload = json.loads(row['payload'])
    if not isinstance(payload, dict):
        raise ValueError(f'El registro RAW {row["id_raw"]} no contiene un objeto JSON')
    payload['id_raw'] = int(row['id_raw'])
    payload['lote_id'] = row['lote_id']
    datos_lista.append(payload)

df_descomprimido = pd.DataFrame(datos_lista)
print(f'Cabeceras descomprimidas: {len(df_descomprimido)}')

Registros descomprimidos exitosamente: 100
Errores de descompresión: 0

Estructura de datos descomprimidos:
  Columnas: ['ExtendedPrice', 'InvoiceID', 'InvoiceLineID', 'LastEditedWhen', 'Quantity', 'StockItemID', 'TaxAmount', 'TaxRate', 'UnitPrice', 'id_raw']

Muestra de datos:


,ExtendedPrice,InvoiceID,InvoiceLineID,LastEditedWhen,Quantity,StockItemID,TaxAmount,TaxRate,UnitPrice,id_raw
0,2645.0,1,1,2013-01-01T12:00:00,10,67,345.0,15.0,-230.0,1
1,134.55,2,2,2013-01-01T12:00:00,9,50,17.55,15.0,13.0,2
2,331.2,2,3,2013-01-01T12:00:00,9,10,43.2,15.0,32.0,3


## 3. Normalizar y transformar datos

In [ ]:
columnas_necesarias = [
    'id_raw', 'lote_id', 'InvoiceID', 'CustomerID', 'BillToCustomerID',
    'OrderID', 'SalespersonPersonID', 'InvoiceDate', 'IsCreditNote',
    'TotalDryItems', 'TotalChillerItems', 'LastEditedWhen'
]
df_aceptados = df_descomprimido.reindex(columns=columnas_necesarias).copy()

for campo in [
    'id_raw', 'InvoiceID', 'CustomerID', 'BillToCustomerID', 'OrderID',
    'SalespersonPersonID', 'TotalDryItems', 'TotalChillerItems'
]:
    df_aceptados[campo] = pd.to_numeric(df_aceptados[campo], errors='raise').astype('Int64')

for campo in ['InvoiceDate', 'LastEditedWhen']:
    df_aceptados[campo] = pd.to_datetime(df_aceptados[campo], errors='raise')

valores_bit = {'true': True, 'false': False, '0': False, '1': True}
es_nota = df_aceptados['IsCreditNote'].astype(str).str.lower().map(valores_bit)
if es_nota.isna().any():
    raise ValueError('Hay valores IsCreditNote invalidos; ejecutar profiling y cuarentena')
df_aceptados['IsCreditNote'] = es_nota.astype(bool)
print(f'Cabeceras normalizadas: {len(df_aceptados)}')
display(df_aceptados.head())

Registros para STAGING: 100

Datos normalizados:


,id_raw,InvoiceID,StockItemID,Quantity
0,1,1,67,10
1,2,2,50,9
2,3,2,10,9
3,4,3,114,3
4,5,4,206,96


## 4. Preparar para guardar en staging

In [ ]:
columnas_mapping = {
    'id_raw': 'id_raw',
    'lote_id': 'lote_id',
    'InvoiceID': 'id_factura',
    'CustomerID': 'id_cliente',
    'BillToCustomerID': 'id_cliente_facturacion',
    'OrderID': 'id_pedido',
    'SalespersonPersonID': 'id_vendedor',
    'InvoiceDate': 'fecha_factura',
    'IsCreditNote': 'es_nota_credito',
    'TotalDryItems': 'total_secos',
    'TotalChillerItems': 'total_refrigerados',
    'LastEditedWhen': 'ultima_edicion'
}
staging_final = df_aceptados[list(columnas_mapping)].rename(columns=columnas_mapping)
print('Estructura final de staging.invoices:')
print(staging_final.dtypes)
display(staging_final.head())

Estructura final para STAGING:
id_linea         int64
id_factura         str
id_producto        str
cantidad           str
neto           float64
impuesto       float64
total          float64
dtype: object

Muestra:


,id_linea,id_factura,id_producto,cantidad,neto,impuesto,total
0,1,1,67,10,2300.0,345.00,2645.00
1,2,2,50,9,117.0,17.55,134.55
2,3,2,10,9,288.0,43.20,331.20
3,4,3,114,3,90.0,13.50,103.50
4,5,4,206,96,259.2,38.88,298.08


## 5. Crear tabla staging.invoices

In [ ]:
create_table_sql = """
IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'staging')
    EXEC('CREATE SCHEMA staging');

IF OBJECT_ID('staging.invoices', 'U') IS NULL
BEGIN
    CREATE TABLE staging.invoices (
        id_raw INT PRIMARY KEY,
        lote_id NVARCHAR(50) NOT NULL,
        id_factura INT NOT NULL,
        id_cliente INT NOT NULL,
        id_cliente_facturacion INT NOT NULL,
        id_pedido INT NULL,
        id_vendedor INT NOT NULL,
        fecha_factura DATE NOT NULL,
        es_nota_credito BIT NOT NULL,
        total_secos INT NOT NULL,
        total_refrigerados INT NOT NULL,
        ultima_edicion DATETIME2(7) NOT NULL,
        timestamp_carga DATETIME DEFAULT GETDATE()
    );
END;
"""

with engine_destino.begin() as conn:
    conn.execute(text(create_table_sql))
print('[OK] Tabla staging.invoices verificada/creada')

Error al crear tabla: Not an executable object: "\nIF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'staging')\n    EXEC('CREATE SCHEMA staging');\n\nIF NOT EXISTS (SELECT * FROM INFORMATION_SCHEMA.TABLES \n              WHERE TABLE_SCHEMA = 'staging' AND TABLE_NAME = 'invoice_lines')\nBEGIN\n    CREATE TABLE staging.invoice_lines (\n        id_linea INT PRIMARY KEY,\n        id_factura INT,\n        id_cliente INT,\n        id_producto INT,\n        cantidad INT,\n        neto DECIMAL(18, 2),\n        impuesto DECIMAL(18, 2),\n        total DECIMAL(18, 2),\n        timestamp_carga DATETIME DEFAULT GETDATE()\n    )\nEND\n"


## 6. Guardar datos en SQL

In [ ]:
sql_retirar_rechazados = """
DELETE FROM staging.invoices
WHERE id_raw IN (
    SELECT r.id_raw
    FROM raw.invoices r
    WHERE EXISTS (
        SELECT 1 FROM quarantine.invoices q
        WHERE q.id_raw = r.id_raw AND q.lote_id = r.lote_id
    ) OR EXISTS (
        SELECT 1 FROM raw.validacion_calidad_invoices v
        WHERE v.id_raw = r.id_raw AND v.lote_id = r.lote_id
          AND v.es_critico = 1
    )
)
"""

with engine_destino.begin() as conn:
    retirados = conn.execute(text(sql_retirar_rechazados)).rowcount
    habilitados = pd.read_sql_query(text(query_raw), conn)['id_raw']
    ids_existentes = pd.read_sql_query(
        'SELECT id_raw FROM staging.invoices', conn
    )['id_raw']
    filas_nuevas = staging_final.drop_duplicates(subset=['id_raw'])
    filas_nuevas = filas_nuevas.loc[
        filas_nuevas['id_raw'].isin(habilitados)
        & ~filas_nuevas['id_raw'].isin(ids_existentes)
    ]
    if not filas_nuevas.empty:
        filas_nuevas.to_sql(
            'invoices', con=conn, schema='staging',
            if_exists='append', index=False
        )

print(f'[OK] {retirados} rechazados retirados; {len(filas_nuevas)} cabeceras nuevas en staging')

[OK] 0 registros nuevos guardados; 100 ya existían.


## 7. Validar datos guardados

In [ ]:
query_control = """
SELECT COUNT(*) AS rechazados_en_staging
FROM staging.invoices s
INNER JOIN raw.invoices r ON r.id_raw = s.id_raw AND r.lote_id = s.lote_id
WHERE EXISTS (
    SELECT 1 FROM quarantine.invoices q
    WHERE q.id_raw = r.id_raw AND q.lote_id = r.lote_id
) OR EXISTS (
    SELECT 1 FROM raw.validacion_calidad_invoices v
    WHERE v.id_raw = r.id_raw AND v.lote_id = r.lote_id
      AND v.es_critico = 1
)
"""
with engine_destino.connect() as conn:
    rechazados = conn.execute(text(query_control)).scalar_one()
if rechazados != 0:
    raise RuntimeError(f'Hay {rechazados} registros rechazados dentro de staging.invoices')
print('[OK] Cero registros de cuarentena o con errores criticos en staging.invoices')

query_validar = """
SELECT COUNT(*) AS total_registros,
       COUNT(DISTINCT id_factura) AS facturas_unicas,
       COUNT(DISTINCT id_cliente) AS clientes_unicos,
       SUM(CAST(es_nota_credito AS INT)) AS notas_credito,
       SUM(CAST(total_secos AS BIGINT)) AS articulos_secos,
       SUM(CAST(total_refrigerados AS BIGINT)) AS articulos_refrigerados
FROM staging.invoices
"""
display(pd.read_sql_query(query_validar, engine_destino))
display(pd.read_sql_query(
    'SELECT TOP 10 * FROM staging.invoices ORDER BY id_raw', engine_destino
))

In [ ]:
auditoria_etapa.finalizar(globals())